# Training Notebook – 5 Klassen

Dieses Notebook erweitert das bestehende Training von 3 auf 5 Klassen.

## Verwendete Klassen
- `cardboard`
- `glass`
- `metal`
- `paper`
- `plastic`

## Wichtig
Es werden **keine neuen Datensatzordner erzeugt**.  
Stattdessen werden die bereits vorhandenen Verzeichnisse verwendet:

- `dataset-split/train`
- `dataset-split/val`
- `dataset-split/test`

Optional kann für das Training auch `dataset-augmented/train` verwendet werden.

## Ziel
Das Notebook lädt nur die 5 gewünschten Klassen, trainiert darauf ein CNN und wertet das Modell anschließend aus.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, Sequential
from sklearn.metrics import classification_report, confusion_matrix


## 1. Projektordner finden

Hier wird nur geprüft, ob das Notebook im richtigen Projektordner läuft.


In [ ]:
DATASET_DIR = "dataset-split"

def find_project_root(start_path, required_folder=DATASET_DIR):
    current = start_path
    while True:
        if required_folder in os.listdir(current):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            return None
        current = parent

print("Aktueller Arbeitsordner:", os.getcwd())

project_root = find_project_root(os.getcwd())

if project_root is None:
    raise FileNotFoundError(f"'{DATASET_DIR}' konnte nicht gefunden werden.")

if os.getcwd() != project_root:
    os.chdir(project_root)
    print("Arbeitsverzeichnis gesetzt auf:", project_root)
else:
    print("Arbeitsverzeichnis war schon korrekt.")

print("Inhalt des Projektordners:")
print(os.listdir())


## 2. Pfade und Klassen festlegen

Standardmäßig wird für das Training der augmentierte Trainingsordner genutzt.
Wenn du ohne Augmentation trainieren willst, setze `USE_AUGMENTED_TRAIN = False`.


In [ ]:
TARGET_CLASSES = ["cardboard", "glass", "metal", "paper", "plastic"]

USE_AUGMENTED_TRAIN = True

if USE_AUGMENTED_TRAIN:
    train_dir = os.path.join("dataset-augmented", "train")
else:
    train_dir = os.path.join("dataset-split", "train")

val_dir = os.path.join("dataset-split", "val")
test_dir = os.path.join("dataset-split", "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 20

print("Train-Verzeichnis:", train_dir)
print("Validation-Verzeichnis:", val_dir)
print("Test-Verzeichnis:", test_dir)
print("Zielklassen:", TARGET_CLASSES)


## 3. Vorhandene Ordner prüfen


In [ ]:
print("train_dir existiert:", os.path.isdir(train_dir))
print("val_dir existiert:", os.path.isdir(val_dir))
print("test_dir existiert:", os.path.isdir(test_dir))

print("\nKlassen in train_dir:", os.listdir(train_dir))
print("Klassen in val_dir:", os.listdir(val_dir))
print("Klassen in test_dir:", os.listdir(test_dir))


## 4. Dateipfade nur für die gewünschten Klassen sammeln

Da in euren Ordnern noch zusätzliche Klassen liegen können, sammeln wir gezielt nur die 5 gewünschten Klassen.


In [ ]:
label_to_index = {label: idx for idx, label in enumerate(TARGET_CLASSES)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

def collect_filepaths(base_dir, class_names):
    filepaths = []
    labels = []

    for cls in class_names:
        cls_dir = os.path.join(base_dir, cls)

        if not os.path.isdir(cls_dir):
            raise FileNotFoundError(f"Klassenordner nicht gefunden: {cls_dir}")

        for file in os.listdir(cls_dir):
            path = os.path.join(cls_dir, file)
            if os.path.isfile(path):
                filepaths.append(path)
                labels.append(label_to_index[cls])

    return pd.DataFrame({
        "filepath": filepaths,
        "label": labels
    })

train_df = collect_filepaths(train_dir, TARGET_CLASSES)
val_df = collect_filepaths(val_dir, TARGET_CLASSES)
test_df = collect_filepaths(test_dir, TARGET_CLASSES)

print("Train:", len(train_df), "Bilder")
print("Val:", len(val_df), "Bilder")
print("Test:", len(test_df), "Bilder")

display(train_df.head())


## 5. Klassenverteilung prüfen


In [ ]:
def class_counts(df, name):
    counts = df["label"].map(index_to_label).value_counts().sort_index()
    print(f"\n{name}")
    print(counts)

class_counts(train_df, "Train")
class_counts(val_df, "Validation")
class_counts(test_df, "Test")


## 6. TensorFlow-Datasets bauen


In [ ]:
def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

def make_dataset(df, training=False):
    paths = df["filepath"].values
    labels = df["label"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)

    ds = ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

print("Datasets erstellt.")


## 7. Beispielbilder anzeigen


In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8).numpy())
        plt.title(index_to_label[int(labels[i].numpy())])
        plt.axis("off")

plt.tight_layout()
plt.show()


In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import MobileNetV2

## 8. CNN-Modell definieren


In [ ]:
# model = Sequential([
#     layers.Input(shape=(224, 224, 3)),
#     layers.Rescaling(1./255),

#     layers.Conv2D(32, (3, 3), activation="relu"),
#     layers.MaxPooling2D(),

#     layers.Conv2D(64, (3, 3), activation="relu"),
#     layers.MaxPooling2D(),

#     layers.Conv2D(128, (3, 3), activation="relu"),
#     layers.MaxPooling2D(),

#     layers.Flatten(),
#     layers.Dense(128, activation="relu"),
#     layers.Dropout(0.3),
#     layers.Dense(len(TARGET_CLASSES), activation="softmax")
# ])

model = keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3,3), padding='same', activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), padding='same', activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), padding='same', activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(256, (3,3), padding='same', activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),

    layers.Dense(6, activation="softmax")

])

model.summary()

## 9. Modell kompilieren


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## RestNet80 Model bauen

In [6]:
RestNet50_model = ResNet50(
   weights='imagenet',       # vortrainiert
   include_top=False,        # ohne alten Klassifikationskopf
   input_shape=(224, 224, 3)
)
for layer in RestNet50_model.layers:
   layer.trainable = False
x = RestNet50_model.output
x = layers.GlobalAveragePooling2D()(x)   # moderner als Flatten
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.4)(x)
output = layers.Dense(6, activation='softmax')(x)
model_resnet = keras.Model(
   inputs=RestNet50_model.input,
   outputs=output
)
model_resnet.summary()
model_resnet.compile(
   optimizer='adam',
   loss='sparse_categorical_crossentropy',
   metrics=['accuracy']
)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_3[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,858,950 (91.01 MB)

 Trainable params: 267,142 (1.02 MB)

 Non-trainable params: 23,591,808 (90.00 MB)

In [7]:
MobileNet_model = MobileNetV2(
   weights='imagenet',
   include_top=False,
   input_shape=(224, 224, 3)
)
for layer in MobileNet_model.layers:
   layer.trainable = False
x = MobileNet_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.4)(x)
output = layers.Dense(6, activation='softmax')(x)
model_mobilenet = keras.Model(
   inputs=MobileNet_model.input,
   outputs=output
)
model_mobilenet.summary()
model_mobilenet.compile(
   optimizer='adam',
   loss='sparse_categorical_crossentropy',
   metrics=['accuracy']
)

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_4[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,427,846 (9.26 MB)

 Trainable params: 167,302 (653.52 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

## 10. Eigenes Modell trainieren


In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)


## 11. Accuracy und Loss visualisieren


In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Training Accuracy")
plt.plot(epochs_range, val_acc, label="Validation Accuracy")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Training Loss")
plt.plot(epochs_range, val_loss, label="Validation Loss")
plt.legend()
plt.title("Loss")

plt.tight_layout()
plt.show()


## 12. Testdaten auswerten


In [ ]:
test_loss, test_acc = model.evaluate(test_ds)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


## 13. Classification Report und Confusion Matrix


In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    preds = np.argmax(preds, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(preds)

print(classification_report(y_true, y_pred, target_names=TARGET_CLASSES))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=TARGET_CLASSES, columns=TARGET_CLASSES)

print(cm_df)

plt.figure(figsize=(7, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(TARGET_CLASSES))
plt.xticks(tick_marks, TARGET_CLASSES, rotation=45)
plt.yticks(tick_marks, TARGET_CLASSES)

for i in range(len(TARGET_CLASSES)):
    for j in range(len(TARGET_CLASSES)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.ylabel("Tatsächliche Klasse")
plt.xlabel("Vorhergesagte Klasse")
plt.tight_layout()
plt.show()


## 14. Beispielvorhersagen

Hier werden Testbilder zusammen mit echter Klasse und Vorhersage angezeigt.


In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in test_ds.take(1):
    preds = model.predict(images, verbose=0)
    preds = np.argmax(preds, axis=1)

    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8).numpy())
        true_label = index_to_label[int(labels[i].numpy())]
        pred_label = index_to_label[int(preds[i])]
        plt.title(f"True: {true_label}\nPred: {pred_label}")
        plt.axis("off")

plt.tight_layout()
plt.show()


## 15. Modell speichern


In [ ]:
model.save("trash_classifier_5classes.keras")
print("Modell gespeichert.")
